In [82]:
from pathlib import Path
import json

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"

ORIGINAL_PATH = RAW_DIR / "multiwoz_original.json"  # adjust filename if needed
ANN1_PATH = PROC_DIR / "ann1.txt"                   # your TSV-like file
ANN2_PATH = PROC_DIR / "ann2.txt"                   # optional later

PROC_DIR.mkdir(parents=True, exist_ok=True)


In [83]:
with ORIGINAL_PATH.open("r", encoding="utf-8") as f:
    original = json.load(f)

original_ids = set(original.keys())
len(original_ids), list(sorted(original_ids))[:5]


(10438,
 ['MUL0001.json',
  'MUL0002.json',
  'MUL0003.json',
  'MUL0004.json',
  'MUL0005.json'])

In [84]:
overlap = original_ids & ann1_ids_raw
len(overlap)


10438

In [85]:
# --- Sanity checks + provenance (Notebook 01) ---

print("=== PATHS (resolved) ===")
print("ORIGINAL_PATH:", ORIGINAL_PATH.resolve())
print("ANN1_PATH:    ", ANN1_PATH.resolve())
print("Exists? original:", ORIGINAL_PATH.exists())
print("Exists? ann1:    ", ANN1_PATH.exists())
print()

print("=== COUNTS ===")
print("Original IDs:   ", len(original_ids))
print("Ann1 IDs (raw): ", len(ann1_ids_raw))
print("Overlap:        ", len(overlap))
print()

# Save the common IDs backbone (for later notebooks)
common_ids = sorted(overlap)
OUT_PATH = PROC_DIR / "common_dialog_ids.json"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(common_ids, f, indent=2)

print("=== SAVED ===")
print("common_dialog_ids.json:", OUT_PATH.resolve())
print("Saved IDs:", len(common_ids))


=== PATHS (resolved) ===
ORIGINAL_PATH: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\raw\multiwoz_original.json
ANN1_PATH:     C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\processed\ann1.txt
Exists? original: True
Exists? ann1:     True

=== COUNTS ===
Original IDs:    10438
Ann1 IDs (raw):  10438
Overlap:         10438

=== SAVED ===
common_dialog_ids.json: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\processed\common_dialog_ids.json
Saved IDs: 10438


In [86]:
def extract_dialog_ids_from_ann_txt(path: Path) -> set[str]:
    """Extract dialog IDs from TSV-like annotation export.

    Expected format per line:
      <utterance text> \\t <json> [\\t <json> ...]
    We parse each JSON chunk and collect `_dialog_index`.
    """
    ids: set[str] = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t")
            # parts[0] = utterance text; parts[1:] = JSON chunks (maybe multiple)
            for chunk in parts[1:]:
                chunk = chunk.strip()
                if not chunk.startswith("{"):
                    continue
                try:
                    obj = json.loads(chunk)
                except json.JSONDecodeError:
                    continue

                d_id = obj.get("_dialog_index")
                if d_id:
                    ids.add(d_id)
    return ids

ann1_ids_raw = extract_dialog_ids_from_ann_txt(ANN1_PATH)
len(ann1_ids_raw), list(sorted(ann1_ids_raw))[:5]



(10438,
 ['MUL0001.json',
  'MUL0002.json',
  'MUL0003.json',
  'MUL0004.json',
  'MUL0005.json'])

In [87]:
ann2_ids = None
if ANN2_PATH.exists():
    ann2_ids_raw = extract_dialog_ids_from_ann_txt(ANN2_PATH)
    ann2_ids = {normalize_dialog_id(x) for x in ann2_ids_raw}
    print("Ann2 IDs:", len(ann2_ids))
else:
    print("Ann2 not found yet (this is fine).")

Ann2 not found yet (this is fine).
